In [0]:

from	delta.tables	import	DeltaTable
from	pyspark.sql	import	functions	as	F
from	pyspark.sql.window	import	Window

In [0]:

#	Seed	Silver	from	the	full	load	(first	run	only)
spark.sql("""
		CREATE	TABLE	IF	NOT	EXISTS	real_estate_dev.silver.properties
		AS	SELECT	*	FROM	real_estate_dev.bronze.properties_full
""")
cdc_df	=	spark.table("real_estate_dev.bronze.properties_cdc")


In [0]:
#	IMPORTANT:	a	property	can	appear	more	than	once	in	a	single	day'sbatch
#	(e.g.	price	change	then	sold).	Keep	only	the	LATEST	change	per	property_id
#	per	run	before	merging,	otherwise	MERGE	will	error	on	duplicate	matches.

In [0]:
w	=	Window.partitionBy("property_id").orderBy(F.col("cdc_timestamp").desc())
cdc_latest	=	(
                cdc_df
				.withColumn("rn",	F.row_number().over(w))\
				.filter("rn	=	1")\
				.drop("rn")
    )

silver_tbl	=	DeltaTable.forName(spark,"real_estate_dev.silver.properties")

(
    silver_tbl.alias("t")
				.merge(cdc_latest.alias("s"),	"t.property_id	=	s.property_id")\
				.whenMatchedDelete(condition="s.operation_type	=	'DELETE'")\
				.whenMatchedUpdateAll(condition="s.operation_type	=	'UPDATE'")\
				.whenNotMatchedInsertAll(condition="s.operation_type	= 'INSERT'")
				.execute()
    )


In [0]:

#	Same	pattern	for	sales_transactions	(INSERT-only,	so	a	simple	append-merge	is	enough

spark.sql("""
		CREATE	TABLE	IF	NOT	EXISTS	real_estate_dev.silver.sales_transactions
		AS	SELECT	*	FROM	real_estate_dev.bronze.sales_transactions_full
    """)
    
txn_cdc	=	spark.table("real_estate_dev.bronze.sales_transactions_cdc")
txn_tbl	=	DeltaTable.forName(spark,	"real_estate_dev.silver.sales_transactions")

(
        txn_tbl.alias("t")
				.merge(txn_cdc.alias("s"),	"t.transaction_id	= s.transaction_id")\
				.whenNotMatchedInsertAll()\
				.execute()
    )